# Exercices SQL Corriges — Data Ingenieur

Ce notebook contient des exercices SQL de base et avances avec leurs corrections.
Les requetes sont ecrites pour **PostgreSQL** (docker-compose fourni).

## Connexion a la base

```bash
# Demarrer les containers
docker compose up -d

# Se connecter en CLI
docker exec -it ms_postgres psql -U dataeng -d sql_avance

# Ou utiliser pgAdmin sur http://localhost:8080
# Email: admin@ms.local / Mot de passe: admin2024
```

---

# PARTIE A — SQL de base (crud_db)

---
### A1. SELECT, WHERE, ORDER BY

**Enonce** : Afficher le nom, prenom et salaire des employes du departement 'IT', tries par salaire decroissant.

In [ ]:
sql_a1 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION A1 =====
sql_a1_correction = """
SELECT nom, prenom, salaire
FROM employes
WHERE departement = 'IT'
ORDER BY salaire DESC;
"""
# Resultat attendu :
# Moreau  | Frank  | 95000
# Petit   | Diana  | 90000
# Martin  | Bob    | 82000
print(sql_a1_correction)

---
### A2. INSERT, UPDATE, DELETE

**Enonce** :
1. Inserer un nouvel employe : 'Nguyen', 'Kim', departement 'IT', salaire 88000
2. Augmenter de 5% le salaire de tous les employes du departement 'RH'
3. Supprimer les employes inactifs (`est_actif = FALSE`)

In [ ]:
sql_a2 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION A2 =====
sql_a2_correction = """
-- 1. INSERT
INSERT INTO employes (nom, prenom, email, departement, salaire, date_embauche)
VALUES ('Nguyen', 'Kim', 'kim.nguyen@ms.com', 'IT', 88000, CURRENT_DATE);

-- 2. UPDATE avec calcul
UPDATE employes
SET salaire = salaire * 1.05
WHERE departement = 'RH';

-- 3. DELETE conditionnel
DELETE FROM employes
WHERE est_actif = FALSE;
"""
print(sql_a2_correction)

---
### A3. JOIN (INNER, LEFT, RIGHT, FULL)

**Enonce** :
1. Lister tous les employes avec le nom de leur(s) projet(s) (meme ceux sans projet)
2. Trouver les projets qui n'ont aucun employe affecte
3. Afficher chaque employe avec son nombre de projets et le total d'heures allouees

In [ ]:
sql_a3 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION A3 =====
sql_a3_correction = """
-- 1. LEFT JOIN : employes avec leurs projets (meme sans projet)
SELECT e.nom, e.prenom, p.nom AS projet, a.role
FROM employes e
LEFT JOIN affectations a ON e.id = a.employe_id
LEFT JOIN projets p ON a.projet_id = p.id
ORDER BY e.nom;

-- 2. Projets sans employes (LEFT JOIN + IS NULL)
SELECT p.nom
FROM projets p
LEFT JOIN affectations a ON p.id = a.projet_id
WHERE a.id IS NULL;

-- 3. Agregation avec JOIN
SELECT
    e.nom,
    e.prenom,
    COUNT(a.projet_id) AS nb_projets,
    COALESCE(SUM(a.heures_allouees), 0) AS total_heures
FROM employes e
LEFT JOIN affectations a ON e.id = a.employe_id
GROUP BY e.id, e.nom, e.prenom
ORDER BY nb_projets DESC;
"""
print(sql_a3_correction)

---
### A4. GROUP BY, HAVING

**Enonce** :
1. Nombre d'employes et salaire moyen par departement
2. Departements ayant un salaire moyen superieur a 75000
3. Pour chaque projet, le nombre d'employes affectes et le total d'heures, uniquement pour les projets avec plus de 2 personnes

In [ ]:
sql_a4 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION A4 =====
sql_a4_correction = """
-- 1. Stats par departement
SELECT
    departement,
    COUNT(*) AS nb_employes,
    ROUND(AVG(salaire), 2) AS salaire_moyen,
    MIN(salaire) AS salaire_min,
    MAX(salaire) AS salaire_max
FROM employes
GROUP BY departement
ORDER BY salaire_moyen DESC;

-- 2. Filtrer avec HAVING
SELECT departement, ROUND(AVG(salaire), 2) AS salaire_moyen
FROM employes
GROUP BY departement
HAVING AVG(salaire) > 75000;

-- 3. Projets avec plus de 2 personnes
SELECT
    p.nom AS projet,
    COUNT(a.employe_id) AS nb_employes,
    SUM(a.heures_allouees) AS total_heures
FROM projets p
JOIN affectations a ON p.id = a.projet_id
GROUP BY p.id, p.nom
HAVING COUNT(a.employe_id) > 2;
"""
print(sql_a4_correction)

---
### A5. Sous-requetes

**Enonce** :
1. Employes dont le salaire est superieur a la moyenne de leur departement
2. Departement(s) ayant le plus grand nombre d'employes
3. Employes travaillant sur tous les projets EN_COURS

In [ ]:
sql_a5 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION A5 =====
sql_a5_correction = """
-- 1. Salaire > moyenne du departement (sous-requete correlee)
SELECT e.nom, e.prenom, e.departement, e.salaire
FROM employes e
WHERE e.salaire > (
    SELECT AVG(e2.salaire)
    FROM employes e2
    WHERE e2.departement = e.departement
)
ORDER BY e.departement, e.salaire DESC;

-- 2. Departement avec le plus d'employes
SELECT departement, COUNT(*) AS nb
FROM employes
GROUP BY departement
HAVING COUNT(*) = (
    SELECT MAX(cnt)
    FROM (SELECT COUNT(*) AS cnt FROM employes GROUP BY departement) sub
);

-- 3. Employes sur TOUS les projets EN_COURS (division relationnelle)
SELECT e.nom, e.prenom
FROM employes e
WHERE NOT EXISTS (
    SELECT p.id
    FROM projets p
    WHERE p.statut = 'EN_COURS'
      AND NOT EXISTS (
          SELECT 1
          FROM affectations a
          WHERE a.employe_id = e.id AND a.projet_id = p.id
      )
);
"""
print(sql_a5_correction)

---
# PARTIE B — SQL Avance (sql_avance)

---
### B1. Window Functions — Classement

**Enonce** : Pour chaque vendeur, afficher :
- Le montant total de ses ventes
- Son rang global par CA (`RANK`)
- Son rang dans sa region (`DENSE_RANK`)
- Le quartile dans lequel il se trouve (`NTILE(4)`)

In [ ]:
sql_b1 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B1 =====
sql_b1_correction = """
WITH ca_vendeur AS (
    SELECT
        vendeur,
        region,
        SUM(montant * quantite) AS ca_total
    FROM ventes
    GROUP BY vendeur, region
)
SELECT
    vendeur,
    region,
    ca_total,
    RANK() OVER (ORDER BY ca_total DESC) AS rang_global,
    DENSE_RANK() OVER (PARTITION BY region ORDER BY ca_total DESC) AS rang_region,
    NTILE(4) OVER (ORDER BY ca_total DESC) AS quartile
FROM ca_vendeur
ORDER BY rang_global;
"""
print(sql_b1_correction)

---
### B2. Window Functions — Cumul et moyenne mobile

**Enonce** : Pour chaque region, afficher par date de vente :
- Le montant de la vente
- Le cumul du montant
- La moyenne mobile sur 3 ventes
- Le montant de la vente precedente (LAG)
- La variation en % par rapport a la vente precedente

In [ ]:
sql_b2 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B2 =====
sql_b2_correction = """
SELECT
    region,
    date_vente,
    produit,
    montant * quantite AS total_vente,

    -- Cumul par region
    SUM(montant * quantite) OVER (
        PARTITION BY region ORDER BY date_vente
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumul,

    -- Moyenne mobile 3 ventes
    ROUND(AVG(montant * quantite) OVER (
        PARTITION BY region ORDER BY date_vente
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moy_mobile_3,

    -- Vente precedente
    LAG(montant * quantite, 1) OVER (
        PARTITION BY region ORDER BY date_vente
    ) AS vente_precedente,

    -- Variation en %
    ROUND(
        (montant * quantite - LAG(montant * quantite, 1) OVER (
            PARTITION BY region ORDER BY date_vente
        )) * 100.0
        / NULLIF(LAG(montant * quantite, 1) OVER (
            PARTITION BY region ORDER BY date_vente
        ), 0)
    , 2) AS variation_pct

FROM ventes
ORDER BY region, date_vente;
"""
print(sql_b2_correction)

---
### B3. CTE Recursive — Hierarchie

**Enonce** : A partir de la table `organisation`, afficher la hierarchie complete :
- Nom, poste, niveau hierarchique (0 pour le PDG)
- Le chemin complet (ex: PDG > VP Tech > Dir Dev > Dev Senior 1)
- Le salaire cumule de chaque sous-arbre

In [ ]:
sql_b3 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B3 =====
sql_b3_correction = """
-- Hierarchie avec chemin
WITH RECURSIVE hierarchie AS (
    -- Ancre : le PDG (manager_id IS NULL)
    SELECT
        id,
        nom,
        poste,
        manager_id,
        salaire,
        0 AS niveau,
        nom::TEXT AS chemin
    FROM organisation
    WHERE manager_id IS NULL

    UNION ALL

    -- Recursion : les subalternes
    SELECT
        o.id,
        o.nom,
        o.poste,
        o.manager_id,
        o.salaire,
        h.niveau + 1,
        h.chemin || ' > ' || o.nom
    FROM organisation o
    JOIN hierarchie h ON o.manager_id = h.id
)
SELECT
    REPEAT('  ', niveau) || nom AS employe,
    poste,
    niveau,
    salaire,
    chemin
FROM hierarchie
ORDER BY chemin;

-- Salaire cumule par sous-arbre (bonus)
WITH RECURSIVE sous_arbre AS (
    SELECT id, nom, manager_id, salaire, id AS racine_id
    FROM organisation

    UNION ALL

    SELECT o.id, o.nom, o.manager_id, o.salaire, s.racine_id
    FROM organisation o
    JOIN sous_arbre s ON o.manager_id = s.id
    WHERE o.id != s.racine_id
)
SELECT
    o.nom,
    o.poste,
    o.salaire AS salaire_propre,
    SUM(s.salaire) AS salaire_sous_arbre
FROM organisation o
JOIN sous_arbre s ON o.id = s.racine_id
GROUP BY o.id, o.nom, o.poste, o.salaire
ORDER BY salaire_sous_arbre DESC;
"""
print(sql_b3_correction)

---
### B4. Gaps and Islands

**Enonce** : A partir de la table `connexions`, identifier pour chaque utilisateur les series de jours consecutifs de connexion. Afficher : user_id, date debut, date fin, nombre de jours consecutifs.

In [ ]:
sql_b4 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B4 =====
sql_b4_correction = """
-- Technique : date - ROW_NUMBER = constante pour un groupe consecutif
WITH numbered AS (
    SELECT
        user_id,
        date_connexion,
        date_connexion - (ROW_NUMBER() OVER (
            PARTITION BY user_id ORDER BY date_connexion
        ))::INT AS grp
    FROM connexions
)
SELECT
    user_id,
    MIN(date_connexion) AS debut_serie,
    MAX(date_connexion) AS fin_serie,
    COUNT(*) AS jours_consecutifs
FROM numbered
GROUP BY user_id, grp
ORDER BY user_id, debut_serie;

-- Resultat attendu :
-- user 1 : 01-03 jan (3j), 06-07 jan (2j), 10-13 jan (4j)
-- user 2 : 01-02 jan (2j), 05-08 jan (4j), 15-16 jan (2j)
-- user 3 : 03-07 jan (5j)
"""
print(sql_b4_correction)

---
### B5. Top-N par groupe

**Enonce** : Afficher les 3 plus grosses transactions par compte, avec :
- Le rang de la transaction dans le compte
- Le % du montant par rapport au total du compte
- Le cumul des montants pour ces top 3

In [ ]:
sql_b5 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B5 =====
sql_b5_correction = """
WITH ranked AS (
    SELECT
        t.id AS txn_id,
        t.compte_id,
        t.montant,
        t.type_transaction,
        t.date_transaction,
        ROW_NUMBER() OVER (
            PARTITION BY t.compte_id ORDER BY ABS(t.montant) DESC
        ) AS rang,
        ROUND(
            ABS(t.montant) * 100.0 / SUM(ABS(t.montant)) OVER (PARTITION BY t.compte_id)
        , 2) AS pct_du_compte
    FROM transactions t
)
SELECT
    compte_id,
    rang,
    montant,
    type_transaction,
    pct_du_compte,
    SUM(ABS(montant)) OVER (
        PARTITION BY compte_id ORDER BY rang
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumul_top
FROM ranked
WHERE rang <= 3
ORDER BY compte_id, rang;
"""
print(sql_b5_correction)

---
### B6. PIVOT avec CASE WHEN

**Enonce** : A partir de `ventes_mensuelles`, creer un tableau croise : une ligne par produit, une colonne par mois (Jan, Fev, Mar) avec le montant, et une colonne Total.

In [ ]:
sql_b6 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B6 =====
sql_b6_correction = """
-- Pivot manuel avec CASE WHEN (fonctionne sur tous les SGBD)
SELECT
    produit,
    SUM(CASE WHEN mois = 'Jan' THEN montant ELSE 0 END) AS jan,
    SUM(CASE WHEN mois = 'Fev' THEN montant ELSE 0 END) AS fev,
    SUM(CASE WHEN mois = 'Mar' THEN montant ELSE 0 END) AS mar,
    SUM(montant) AS total
FROM ventes_mensuelles
GROUP BY produit
ORDER BY total DESC;

-- Resultat attendu :
-- Serveur  | 45000 | 38000 | 52000 | 135000
-- Laptop   | 15000 | 18000 | 22000 |  55000
-- Ecran    |  8000 |  9500 | 11000 |  28500
-- Clavier  |  2000 |  2500 |  3000 |   7500
"""
print(sql_b6_correction)

---
### B7. Requete complexe multi-tables

**Enonce** : Ecrire UNE seule requete retournant pour chaque client :
- Son nom et segment
- Le nombre de comptes actifs
- Le montant total des transactions du dernier mois de donnees
- Son rang par montant total dans son segment
- Uniquement les clients dans le top 5 de leur segment

In [ ]:
sql_b7 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B7 =====
sql_b7_correction = """
WITH derniere_date AS (
    SELECT MAX(date_transaction)::DATE AS max_date FROM transactions
),
stats_client AS (
    SELECT
        c.id AS client_id,
        c.nom,
        c.segment,
        COUNT(DISTINCT CASE WHEN co.statut = 'ACTIF' THEN co.id END) AS nb_comptes_actifs,
        COALESCE(SUM(
            CASE WHEN t.date_transaction >= (SELECT max_date - INTERVAL '30 days' FROM derniere_date)
                 THEN ABS(t.montant)
                 ELSE 0
            END
        ), 0) AS montant_30j
    FROM clients c
    LEFT JOIN comptes co ON c.id = co.client_id
    LEFT JOIN transactions t ON co.id = t.compte_id
    GROUP BY c.id, c.nom, c.segment
),
ranked AS (
    SELECT
        *,
        RANK() OVER (PARTITION BY segment ORDER BY montant_30j DESC) AS rang_segment
    FROM stats_client
)
SELECT
    nom,
    segment,
    nb_comptes_actifs,
    ROUND(montant_30j, 2) AS montant_30j,
    rang_segment
FROM ranked
WHERE rang_segment <= 5
ORDER BY segment, rang_segment;
"""
print(sql_b7_correction)

---
### B8. Procedures stockees et Triggers

**Enonce** :
1. Creer une procedure `sp_augmentation(p_departement VARCHAR, p_pourcentage DECIMAL)` qui augmente le salaire de tous les employes d'un departement
2. Creer un trigger qui empeche la suppression d'un employe ayant des affectations actives
3. Creer une vue materialisee du CA par region et par mois

In [ ]:
sql_b8 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B8 =====
sql_b8_correction = """
-- 1. Procedure d'augmentation
CREATE OR REPLACE PROCEDURE sp_augmentation(
    p_departement VARCHAR,
    p_pourcentage DECIMAL
)
LANGUAGE plpgsql AS $$
DECLARE
    v_nb INT;
BEGIN
    UPDATE employes
    SET salaire = salaire * (1 + p_pourcentage / 100)
    WHERE departement = p_departement;

    GET DIAGNOSTICS v_nb = ROW_COUNT;
    RAISE NOTICE '% employes mis a jour dans %', v_nb, p_departement;
END;
$$;
-- Appel : CALL sp_augmentation('IT', 5.0);

-- 2. Trigger de protection
CREATE OR REPLACE FUNCTION fn_protect_employe()
RETURNS TRIGGER AS $$
BEGIN
    IF EXISTS (SELECT 1 FROM affectations WHERE employe_id = OLD.id) THEN
        RAISE EXCEPTION 'Impossible de supprimer : employe % a des affectations actives', OLD.nom;
    END IF;
    RETURN OLD;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER trg_protect_employe
    BEFORE DELETE ON employes
    FOR EACH ROW
    EXECUTE FUNCTION fn_protect_employe();

-- 3. Vue materialisee
CREATE MATERIALIZED VIEW mv_ca_region_mois AS
SELECT
    region,
    DATE_TRUNC('month', date_vente)::DATE AS mois,
    SUM(montant * quantite) AS ca_total,
    COUNT(*) AS nb_ventes,
    AVG(montant * quantite) AS ca_moyen
FROM ventes
GROUP BY region, DATE_TRUNC('month', date_vente)
ORDER BY region, mois;

-- Rafraichir : REFRESH MATERIALIZED VIEW mv_ca_region_mois;
"""
print(sql_b8_correction)

---
### B9. ETL en SQL — MERGE / Upsert

**Enonce** : Ecrire un MERGE qui synchronise `staging.raw_clients` vers `dwh.dim_client` en SCD Type 2 :
- Mettre a jour (fermer l'ancienne ligne + inserer la nouvelle) si ville ou segment a change
- Inserer les nouveaux clients

*(A executer sur la base `etl_warehouse`)*

In [ ]:
sql_b9 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B9 =====
sql_b9_correction = """
-- PostgreSQL n'a pas MERGE natif avant v15, on utilise une transaction
-- Sur PostgreSQL >= 15 on peut utiliser MERGE

BEGIN;

-- Etape 1 : Fermer les lignes qui ont change
UPDATE dwh.dim_client dc
SET
    date_fin = CURRENT_DATE - INTERVAL '1 day',
    est_courant = FALSE
FROM staging.raw_clients rc
WHERE dc.client_id = rc.client_id
  AND dc.est_courant = TRUE
  AND (dc.ville IS DISTINCT FROM rc.ville
       OR dc.segment IS DISTINCT FROM rc.segment);

-- Etape 2 : Inserer les nouvelles versions + nouveaux clients
INSERT INTO dwh.dim_client (client_id, nom, email, ville, pays, segment, date_debut)
SELECT
    rc.client_id,
    rc.nom,
    rc.email,
    rc.ville,
    rc.pays,
    rc.segment,
    CURRENT_DATE
FROM staging.raw_clients rc
WHERE NOT EXISTS (
    SELECT 1 FROM dwh.dim_client dc
    WHERE dc.client_id = rc.client_id
      AND dc.est_courant = TRUE
);

COMMIT;

-- Alternative avec MERGE (PostgreSQL 15+) :
/*
MERGE INTO dwh.dim_client dc
USING staging.raw_clients rc ON dc.client_id = rc.client_id AND dc.est_courant = TRUE
WHEN MATCHED AND (dc.ville IS DISTINCT FROM rc.ville OR dc.segment IS DISTINCT FROM rc.segment) THEN
    UPDATE SET date_fin = CURRENT_DATE - 1, est_courant = FALSE
WHEN NOT MATCHED THEN
    INSERT (client_id, nom, email, ville, pays, segment, date_debut)
    VALUES (rc.client_id, rc.nom, rc.email, rc.ville, rc.pays, rc.segment, CURRENT_DATE);
*/
"""
print(sql_b9_correction)

---
### B10. Analyse avancee — Requete financiere

**Enonce** : Ecrire une requete qui calcule pour chaque compte :
- Le solde courant (cumul des credits - debits)
- La transaction la plus recente
- Le nombre de jours sans transaction
- La moyenne mobile du montant sur les 5 dernieres transactions
- Si le compte est en "alerte" (plus de 3 debits consecutifs)

In [ ]:
sql_b10 = """
-- Votre requete ici

"""

In [ ]:
# ===== CORRECTION B10 =====
sql_b10_correction = """
WITH soldes AS (
    SELECT
        compte_id,
        SUM(CASE
            WHEN type_transaction = 'CREDIT' THEN montant
            WHEN type_transaction IN ('DEBIT', 'PRELEVEMENT') THEN -montant
            ELSE 0
        END) AS solde_courant,
        MAX(date_transaction) AS derniere_txn,
        COUNT(*) AS nb_transactions
    FROM transactions
    GROUP BY compte_id
),
debits_consecutifs AS (
    SELECT
        compte_id,
        type_transaction,
        date_transaction,
        -- Grouper les debits consecutifs
        ROW_NUMBER() OVER (PARTITION BY compte_id ORDER BY date_transaction)
        - ROW_NUMBER() OVER (
            PARTITION BY compte_id, 
            CASE WHEN type_transaction IN ('DEBIT', 'PRELEVEMENT') THEN 1 ELSE 0 END
            ORDER BY date_transaction
        ) AS grp
    FROM transactions
    WHERE type_transaction IN ('DEBIT', 'PRELEVEMENT')
),
max_consecutifs AS (
    SELECT compte_id, MAX(cnt) AS max_debits_consec
    FROM (
        SELECT compte_id, grp, COUNT(*) AS cnt
        FROM debits_consecutifs
        GROUP BY compte_id, grp
    ) sub
    GROUP BY compte_id
),
moy_mobile AS (
    SELECT DISTINCT ON (compte_id)
        compte_id,
        AVG(montant) OVER (
            PARTITION BY compte_id ORDER BY date_transaction DESC
            ROWS BETWEEN CURRENT ROW AND 4 FOLLOWING
        ) AS moy_5_dernieres
    FROM transactions
)
SELECT
    s.compte_id,
    co.type_compte,
    ROUND(s.solde_courant, 2) AS solde_courant,
    s.derniere_txn,
    CURRENT_DATE - s.derniere_txn::DATE AS jours_sans_txn,
    s.nb_transactions,
    ROUND(m.moy_5_dernieres, 2) AS moy_mobile_5,
    COALESCE(mc.max_debits_consec, 0) AS max_debits_consecutifs,
    CASE WHEN COALESCE(mc.max_debits_consec, 0) > 3 THEN 'ALERTE' ELSE 'OK' END AS statut_alerte
FROM soldes s
JOIN comptes co ON s.compte_id = co.id
LEFT JOIN max_consecutifs mc ON s.compte_id = mc.compte_id
LEFT JOIN moy_mobile m ON s.compte_id = m.compte_id
ORDER BY s.compte_id;
"""
print(sql_b10_correction)

---

## Aide-memoire : Commandes utiles

```bash
# Demarrer
docker compose up -d

# Se connecter a une base
docker exec -it ms_postgres psql -U dataeng -d crud_db
docker exec -it ms_postgres psql -U dataeng -d sql_avance
docker exec -it ms_postgres psql -U dataeng -d etl_warehouse

# Executer un script SQL
docker exec -it ms_postgres psql -U dataeng -d sql_avance -f /sql_scripts/procedures_triggers.sql

# Importer un CSV
docker exec -it ms_postgres psql -U dataeng -d etl_warehouse -f /sql_scripts/import_csv.sql

# Exporter vers CSV
docker exec -it ms_postgres psql -U dataeng -d etl_warehouse -f /sql_scripts/export_csv.sql

# Lister les bases
docker exec -it ms_postgres psql -U dataeng -c '\l'

# Lister les tables d'une base
docker exec -it ms_postgres psql -U dataeng -d sql_avance -c '\dt'

# Arreter
docker compose down

# Arreter et supprimer les donnees
docker compose down -v
```